# Prétraitement du dataset Diabète – Sprint 1

Ce notebook couvre :
1. Lecture et inspection des données
2. Prétraitement initial (regroupement des classes)
3. Séparation train / validation / test
4. Normalisation des features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Lecture du dataset
On charge le fichier CSV et on affiche un aperçu ainsi que les informations générales.

In [ ]:
def read_data(file_path="../data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"):
    try:
        df = pd.read_csv(file_path)
        print("Aperçu des 5 premières lignes :")
        print(df.head())
        print("\nInformations générales :")
        print(df.info())
        print("\nValeurs manquantes par colonne :")
        print(df.isnull().sum())
        df.to_parquet("test_index_false.parquet", index=False)
        return df
    except FileNotFoundError:
        print(f"Erreur : le fichier {file_path} n'a pas été trouvé.")
        return None

df = read_data()

## Prétraitement initial
- Regroupement des classes : 0 = no diabète ou prediabète, 1 = diabète
- Séparation des features (X) et de la cible (y)

In [ ]:
def separate_data(df):
    X = df.drop("Diabetes_binary", axis=1)
    y = df["Diabetes_binary"]
    return X, y

if df is not None:
    X, y = separate_data(df)
    print("Distribution des classes :")
    print(y.value_counts())

## Séparation Train / Validation / Test
- 70% train
- 15% validation
- 15% test
- Stratification pour garder la proportion des classes

In [ ]:
if df is not None:
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    
    print("Taille train :", X_train.shape)
    print("Taille validation :", X_val.shape)
    print("Taille test :", X_test.shape)
    print(f"nb de ligne : {len(df)}")

## Normalisation des features
- Standardisation : moyenne=0, écart type=1
- Fit sur le train, transform sur validation et test

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Normalisation effectuée. Exemple :")
print(X_train_scaled[:5])

## Sauvegarde des datasets prétraités

Les datasets sont sauvegardés dans :

- `data/raw` : données brutes
- `data/processed` : données prétraitées

On sauvegarde :
- train
- validation
- test
- le scaler utilisé pour la normalisation

Cela permet de reproduire exactement le pipeline de preprocessing.

In [ ]:
import os
import pandas as pd
import joblib

# Création des dossiers projet
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

# Reconstruction des DataFrames à partir des données normalisées
df_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
df_train_scaled["Diabetes_binary"] = y_train.reset_index(drop=True)

df_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
df_val_scaled["Diabetes_binary"] = y_val.reset_index(drop=True)

df_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
df_test_scaled["Diabetes_binary"] = y_test.reset_index(drop=True)

# Sauvegarde des datasets
df_train_scaled.to_parquet("data/processed/train.parquet", index=False)
df_val_scaled.to_parquet("data/processed/val.parquet", index=False)
df_test_scaled.to_parquet("data/processed/test.parquet", index=False)

# Sauvegarde du scaler
joblib.dump(scaler, "artifacts/scaler.joblib")

print("Datasets sauvegardés dans data/processed/")
print("Scaler sauvegardé dans artifacts/")